# S07 · 02 — Observabilidad del servicio: Prometheus de verdad

**Objetivo.** Entender qué mide Prometheus, qué tipo de métrica usar en cada caso,
cómo se calcula un p95 a partir de buckets y por qué la elección de buckets decide si
ese p95 significa algo.

**La distinción que sostiene toda la sesión:**

- **Prometheus mide el SERVICIO**: latencia, throughput, errores, saturación.
  Continuo, barato, alertas en minutos.
- **Evidently mide los DATOS y el MODELO**: distribuciones, drift, calidad. Por lotes,
  fuera del request path, alertas en horas o días.

Un dashboard verde de Prometheus **no** significa que el modelo funcione: el servicio
puede devolver basura en 8 ms con cero errores HTTP.

**Requisitos.** Nada externo: `prometheus_client` trae su propio registro en memoria y
todo este notebook corre sin levantar servidores.

In [ ]:
from prometheus_client import (
    CollectorRegistry,
    Counter,
    Gauge,
    Histogram,
    Summary,
    generate_latest,
)

# Registro propio, NO el global (`REGISTRY`). Motivo: registrar dos veces una
# metrica con el mismo nombre en el registro global lanza
# `Duplicated timeseries in CollectorRegistry`, y en un notebook que se re-ejecuta
# celda por celda eso pasa siempre. Un registro local se puede recrear.
registro = CollectorRegistry()
print("registro limpio:", registro)

## 1. Los cuatro tipos de métrica

| Tipo | Qué representa | Solo sube? | Preguntas que contesta | PromQL típico |
|---|---|---|---|---|
| **Counter** | total acumulado de eventos | Sí (monótono; se reinicia a 0 si el proceso reinicia) | ¿cuántas predicciones? ¿cuántos errores? ¿a qué ritmo? | `rate(x_total[5m])` |
| **Gauge** | valor instantáneo que sube y baja | No | ¿cuántos requests en vuelo? ¿qué versión está cargada? | `x`, `max_over_time(x[1h])` |
| **Histogram** | observaciones repartidas en buckets acumulativos | Sí (cada bucket es un counter) | ¿cuál es el p95 de latencia? ¿qué fracción bajo 100 ms? | `histogram_quantile(0.95, ...)` |
| **Summary** | cuantiles calculados **en el proceso** | Sí | igual que Histogram, pero... | no se puede agregar entre instancias |

**Por qué el curso usa Histogram y no Summary.** Los cuantiles de un `Summary` se
calculan dentro de cada proceso. Con tres réplicas de la API detrás de un balanceador
tendrías tres p95 y **no existe forma matemática de combinarlos** en el p95 global (el
promedio de tres percentiles no es un percentil). El `Histogram` expone los buckets
crudos, que sí se suman entre réplicas, y el cuantil se calcula al consultar. El
precio es que el resultado es una **interpolación**, no el valor exacto.

Regla práctica: si la métrica se va a agregar entre instancias —y en un servicio con
más de una réplica siempre se va a agregar— usa `Histogram`.

In [ ]:
# Un ejemplo de cada uno, en el registro local.
peticiones = Counter(
    "demo_peticiones_total",
    "Peticiones atendidas.",
    ["ruta", "codigo"],
    registry=registro,
)
en_vuelo = Gauge("demo_en_vuelo", "Peticiones en curso.", registry=registro)
latencia = Histogram(
    "demo_latencia_segundos",
    "Latencia de la peticion, en segundos.",
    buckets=(0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0),
    registry=registro,
)
resumen = Summary("demo_resumen_segundos", "Ejemplo de Summary.", registry=registro)

# Convenciones de nombres (son parte del contrato operativo, se consultan en PromQL):
#   - unidad base en el nombre: _segundos, nunca milisegundos;
#   - sufijo _total en los counters;
#   - labels de cardinalidad ACOTADA: decenas de valores, no miles.
print("metricas creadas")

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# Trafico simulado: mezcla log-normal (el cuerpo, ~4 ms, que es el orden de una
# inferencia sklearn en CPU) mas una cola lenta (el 3% que duele). Toda
# distribucion de latencia real tiene esta forma: asimetrica y con cola larga.
muestras = np.concatenate(
    [
        rng.lognormal(mean=np.log(0.004), sigma=0.40, size=970),
        rng.lognormal(mean=np.log(0.045), sigma=0.55, size=30),
    ]
)

for segundos in muestras:
    en_vuelo.inc()
    latencia.observe(segundos)
    resumen.observe(segundos)
    peticiones.labels(ruta="/predict", codigo="200").inc()
    en_vuelo.dec()

peticiones.labels(ruta="/predict", codigo="422").inc(11)
print(f"observaciones: {len(muestras)}  media: {muestras.mean() * 1000:.1f} ms")
print(f"p50 real: {np.percentile(muestras, 50) * 1000:.1f} ms")
print(f"p95 real: {np.percentile(muestras, 95) * 1000:.1f} ms")

## 2. Qué expone realmente `/metrics`

El formato de exposición es texto plano. Vale la pena leerlo entero una vez: casi
todos los malentendidos sobre histogramas se resuelven mirando estas líneas.

Fíjate en tres cosas:

1. El `Histogram` genera **tres** familias de series: `_bucket`, `_sum` y `_count`.
2. Los buckets son **acumulativos**: `le="0.05"` cuenta *todas* las observaciones
   menores o iguales a 0.05, no las que caen entre 0.025 y 0.05.
3. Existe un bucket `le="+Inf"` que iguala a `_count`. Es lo que permite calcular
   fracciones.

In [ ]:
texto = generate_latest(registro).decode()
for linea in texto.splitlines():
    if linea.startswith("demo_latencia") or linea.startswith("demo_peticiones"):
        print(linea)

## 3. Cómo se calcula un p95 a partir de buckets

`histogram_quantile` de PromQL hace exactamente esto:

1. Busca el bucket donde el conteo acumulado cruza el percentil buscado
   (`0.95 * total`).
2. **Interpola linealmente** dentro de ese bucket, asumiendo que las observaciones se
   reparten de forma uniforme entre su borde inferior y su borde superior.

De ahí se siguen dos consecuencias que hay que tener presentes al leer un dashboard:

- el p95 reportado **nunca** es exacto: es una estimación limitada por los bordes;
- si el percentil cae en un bucket muy ancho, el error puede ser enorme;
- si cae en el bucket `+Inf`, `histogram_quantile` devuelve el borde del último
  bucket finito y el valor real puede ser cualquier cosa por encima.

Vamos a implementarlo a mano y comparar con el percentil real.

In [ ]:
def cuantil_desde_buckets(bordes: list[float], acumulados: list[float], q: float) -> float:
    """Reimplementacion didactica de histogram_quantile de PromQL.

    bordes y acumulados vienen ordenados; acumulados es monotono no decreciente.
    """
    total = acumulados[-1]
    if total == 0:
        return float("nan")
    objetivo = q * total
    anterior_borde, anterior_cuenta = 0.0, 0.0
    for borde, cuenta in zip(bordes, acumulados):
        if cuenta >= objetivo:
            if borde == float("inf"):
                # PromQL devuelve el ultimo borde finito. La cola queda invisible.
                return anterior_borde
            if cuenta == anterior_cuenta:
                return borde
            fraccion = (objetivo - anterior_cuenta) / (cuenta - anterior_cuenta)
            return anterior_borde + fraccion * (borde - anterior_borde)
        anterior_borde, anterior_cuenta = borde, cuenta
    return bordes[-1]


def leer_buckets(registro: CollectorRegistry, metrica: str) -> tuple[list[float], list[float]]:
    """Extrae (bordes, acumulados) de un Histogram del registro."""
    bordes, acumulados = [], []
    for familia in registro.collect():
        if familia.name != metrica:
            continue
        for muestra in familia.samples:
            if muestra.name.endswith("_bucket"):
                bordes.append(float(muestra.labels["le"]))
                acumulados.append(muestra.value)
    orden = np.argsort(bordes)
    return [bordes[i] for i in orden], [acumulados[i] for i in orden]


bordes, acumulados = leer_buckets(registro, "demo_latencia_segundos")
for q in (0.5, 0.9, 0.95, 0.99):
    estimado = cuantil_desde_buckets(bordes, acumulados, q)
    real = float(np.percentile(muestras, q * 100))
    print(
        f"p{int(q * 100):>2}  buckets: {estimado * 1000:7.1f} ms   "
        f"real: {real * 1000:7.1f} ms   error: {100 * (estimado / real - 1):+6.1f}%"
    )

## 4. Los buckets mal elegidos son el error más común

Los buckets por defecto de `prometheus_client` van de 5 ms a 10 s. Para una inferencia
sklearn en CPU, que vive en el orden de 1-20 ms, eso significa que **casi todo cae en
el primer bucket** y el p95 pierde toda resolución justo donde importa.

Compáralo con los buckets afinados para este servicio, que son los que están en
`taxi.api.metricas`. Mira **las dos columnas de error**, no solo la del p95: los dos
juegos de buckets comparten bordes alrededor del p95, así que ahí empatan, y la
diferencia aparece en el p50, donde los buckets por defecto tienen un solo escalón
por debajo de 5 ms. Con menos buckets (11 en lugar de 14) los afinados dan más
resolución donde vive este servicio. Ejecuta la celda y compara los números que te
salgan.

In [ ]:
BUCKETS_DEFECTO = (0.005, 0.01, 0.025, 0.05, 0.075, 0.1, 0.25, 0.5, 0.75, 1.0, 2.5, 5.0, 7.5, 10.0)
BUCKETS_AFINADOS = (0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0, 2.5)
BUCKETS_MALOS = (1.0, 5.0, 10.0)  # todo cae en el primero: el p95 no dice nada

reales = {q: float(np.percentile(muestras, q * 100)) for q in (0.5, 0.95)}
filas = []
for nombre, buckets in [
    ("por defecto", BUCKETS_DEFECTO),
    ("afinados (curso)", BUCKETS_AFINADOS),
    ("mal elegidos", BUCKETS_MALOS),
]:
    reg = CollectorRegistry()
    h = Histogram("t_segundos", "tmp", buckets=buckets, registry=reg)
    for s in muestras:
        h.observe(s)
    b, a = leer_buckets(reg, "t_segundos")
    fila = {"buckets": nombre, "n": len(buckets)}
    for q in (0.5, 0.95):
        estimado = cuantil_desde_buckets(b, a, q)
        fila[f"p{int(q * 100)}_ms"] = round(estimado * 1000, 2)
        fila[f"error_p{int(q * 100)}_%"] = round(100 * (estimado / reales[q] - 1), 1)
    filas.append(fila)

for q, valor in reales.items():
    print(f"p{int(q * 100)} real = {valor * 1000:.2f} ms")
print()
for fila in filas:
    print(fila)

**Cómo elegir buckets.** Tres criterios, en orden:

1. Que el **objetivo de servicio (SLO)** sea un borde exacto. Si prometes "p95 bajo 50
   ms", tiene que existir `le="0.05"`; así `sum(rate(..._bucket{le="0.05"}[5m])) /
   sum(rate(..._count[5m]))` da directamente la fracción que cumple el SLO, sin
   interpolar.
2. Espaciado aproximadamente logarítmico alrededor de la latencia típica.
3. Pocos buckets: cada uno es una serie temporal más, multiplicada por cada
   combinación de labels. 10-12 es un número razonable; 40 no.

Y un aviso: **cambiar los buckets rompe la comparabilidad histórica** del cuantil.
No es un cambio cosmético.

## 5. La instrumentación real del curso

Ya está escrita, en [`src/taxi/api/metricas.py`](../../../src/taxi/api/metricas.py). No
la dupliques: lee lo que hay y entiende por qué cada decisión está tomada así.

In [ ]:
from taxi.api import metricas

print(metricas.__doc__.split("Convenciones")[0])
print("tipos de error declarados (conjunto CERRADO):", metricas.TIPOS_ERROR)

In [ ]:
# Se simula un poco de trafico sobre las metricas REALES del servicio y se mira
# la exposicion. Cuidado: esto escribe en el registro global del proceso.
metricas.fijar_modelo("nyc-taxi-duration", "7", "models:/nyc-taxi-duration@champion")
for s in muestras[:200]:
    metricas.observar_latencia(version="7", segundos=float(s))
    metricas.registrar_prediccion(version="7", viaje_largo=bool(s > 0.02))
metricas.registrar_error("validacion")

from prometheus_client import REGISTRY

for linea in generate_latest(REGISTRY).decode().splitlines():
    if linea.startswith("taxi_") and not linea.startswith("taxi_inferencia_duracion_segundos_bucket"):
        print(linea)

Tres decisiones de ese módulo que merecen atención, porque son las que se olvidan en
la práctica:

1. **Pre-inicializar las series de error en 0.** Un `Counter` con labels no existe en
   `/metrics` hasta la primera llamada a `.labels(...)`. Antes del primer error,
   `rate(taxi_errores_total[5m])` no devuelve `0`: no devuelve **nada**, y el panel
   muestra "No data", que es indistinguible de "el exporter está caído".
2. **`Gauge` con `.clear()` para la info del modelo**, en lugar del tipo `Info`. Al
   promover un modelo hay que dejar de reportar la versión anterior; con `Info`
   quedarían dos series activas y las consultas por `model_version` devolverían dos
   valores.
3. **`tipo` de error como conjunto cerrado.** Usar el nombre de la excepción como
   label deja que una librería de terceros decida la cardinalidad de tus métricas, y
   además puede filtrar detalles internos a un endpoint público.

## 6. Del `/metrics` al dashboard

El dashboard ya está versionado como JSON y se provisiona automáticamente. No se
construye a mano en la UI de Grafana: lo que solo existe en la base de datos de
Grafana no se revisa en un PR y se pierde al reinstalar.

In [ ]:
import json

from taxi.config import PROJECT_ROOT

ruta = PROJECT_ROOT / "observabilidad" / "grafana" / "dashboards" / "api-modelo.json"
dashboard = json.loads(ruta.read_text(encoding="utf-8"))
print(dashboard["title"], "| uid:", dashboard.get("uid"), "\n")

for panel in dashboard["panels"]:
    print(f"[{panel['type']}] {panel['title']}")
    for objetivo in panel.get("targets", []) or []:
        print("    ", objetivo.get("expr"))

Lee esas expresiones con calma; son el vocabulario mínimo de PromQL para un servicio
de ML:

| Expresión | Qué contesta |
|---|---|
| `rate(taxi_predicciones_total[$__rate_interval])` | throughput (predicciones por segundo) |
| `histogram_quantile(0.95, sum by (le) (rate(..._bucket[...])))` | p95 de latencia. `sum by (le)` **antes** de `histogram_quantile`: agregar cuantiles ya calculados no es válido |
| `sum by (tipo) (rate(taxi_errores_total[...]))` | errores por tipo, no un total opaco |
| `sum by (clase) (rate(taxi_predicciones_total[...]))` | **prediction drift observable en tiempo real**: si la proporción de `largo` pasa de 20% a 60% de un día para otro, algo cambió en la entrada aunque la latencia siga perfecta |
| `taxi_modelo_info` | qué versión responde. Es lo que permite atribuir un cambio de latencia a un despliegue |

La cuarta fila es el puente entre las dos mitades de esta sesión: es una métrica de
Prometheus que habla del modelo, no del servicio. Es barata, está disponible al
instante y **no sustituye** al check de drift; lo complementa como señal temprana.

Para levantarlo todo:

```bash
docker compose up -d          # API + Prometheus + Grafana
curl -s localhost:8000/metrics | head -40
# Grafana en localhost:3000, dashboard "API de inferencia — modelo de duracion"
```

## 7. Ejercicios

1. **Añade una métrica que falta.** El servicio no expone el **tamaño del lote** de
   `/predict/lote`. ¿`Counter`, `Gauge` o `Histogram`? Justifica con la pregunta que
   quieres contestar ("¿cuántas filas por lote nos manda el cliente?") y añádela a
   `taxi.api.metricas`.
2. **Un SLO real.** Elige un objetivo de latencia p95 para `/predict` y verifica que
   existe un borde de bucket exacto en ese valor. Escribe la consulta PromQL que
   devuelve la fracción de requests que cumplen el SLO.
3. **Rompe la cardinalidad a propósito.** Añade `PU_DO` como label en un registro
   local, genera 2.000 valores distintos y cuenta las series con
   `len(generate_latest(reg).decode().splitlines())`. Extrapola a 265 zonas × 265
   destinos × 3 versiones de modelo y explica por qué eso tumba el servidor.
4. **Alerta.** Escribe la regla de Prometheus (`alert`, `expr`, `for`) para "el p95 de
   inferencia supera 100 ms durante 5 minutos". Piensa en el `for`: sin él, un pico de
   un scrape genera una página a las 3 a.m.